In [1]:
import pandas as pd
import numpy as np

## Data Aquisition

In [2]:
path = "../data/df_model.csv"

df = pd.read_csv(path)

df.head()

,primary_energy_consumption,population,gdp,coal_consumption,oil_consumption,gas_consumption,fossil_fuel_consumption,solar_consumption,wind_consumption,hydro_consumption,renewables_consumption,electricity_generation,hydro_share_energy
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,368.65,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,397.19,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,422.82,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,447.15,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,484.94,NaN


## Splitting

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [4]:
# Drop rows where the target variable 'primary_energy_consumption' is NaN
df_cleaned = df.dropna(subset=["primary_energy_consumption"])

X = df_cleaned.drop("primary_energy_consumption", axis=1)
y = df_cleaned["primary_energy_consumption"]

# shuffle, 1/3
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size=1/3, random_state=42)

In [5]:
num_attribs = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_attribs = X_train.select_dtypes(include='object').columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('std_scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

# Fit and transform based on the current X_train and X_test
X_train_prepared = full_pipeline.fit_transform(X_train, y_train)
X_test_prepared = full_pipeline.transform(X_test)

print(f"New X_train_prepared shape: {X_train_prepared.shape}")
print(f"New y_train shape: {y_train.shape}")

New X_train_prepared shape: (8392, 12)
New y_train shape: (8392,)


# Model

ANN key parts:
- weighted sum and bias
- sigmoid activation function
- matrix form of ANN computation
- (Structure) input layer, hidden layer, output layer architecture
- full forward propagation

### ANN
Inputs:
- learning rate (0.001 - 0.01)
- hidden_layer_sizes
- output size

Weighted Sum and Bias:

z = xW+b


## Training
- forward proagation
- loss
- back propagation
- gradient descent

Inputs:
- features
- target
- epochs (number of loops)


In [6]:
class ANN:
  def __init__(self, input_size, hidden_layer_size, output_size, lr):
    self.lr = lr

    # weighted sum and bias
    self.W1 = np.random.randn(input_size, hidden_layer_size) * 0.01
    self.b1 = np.zeros((1, hidden_layer_size))

    self.W2 = np.random.randn(hidden_layer_size, output_size) * 0.01
    self.b2 = np.zeros((1, output_size))

  def forward_prop(self, X):
    # Hidden layer calculation
    self.z1 = np.dot(X, self.W1) + self.b1
    self.a1 = np.maximum(0, self.z1)   # ReLU activation

    # Output layer calculation
    self.z2 = np.dot(self.a1, self.W2) + self.b2

    return self.z2

  def compute_loss(self, y_true, y_pred):
    loss = np.mean((y_true - y_pred) ** 2)
    return loss

  def backpropagation(self, X, y, y_pred):
    m = X.shape[0] # Define m as the number of samples

    # output layer error
    dz2 = y_pred - y

    # output layer gradients
    dW2 = (self.a1.T @ dz2) / m
    db2 = np.mean(dz2, axis=0)

    # hidden layer error (using ReLU derivative since forward_prop uses ReLU)
    d_relu = (self.z1 > 0).astype(float) # Derivative of ReLU is 1 if z > 0, else 0
    dz1 = (dz2 @ self.W2.T) * d_relu

    # hidden layer gradients
    dW1 = (X.T @ dz1) / m
    db1 = np.mean(dz1, axis=0)

    return dW1, db1, dW2, db2

  def training(self, X, y, epochs = 100):
    y = y.values.reshape(-1, 1) # Ensure y is a column vector
    for epoch in range(epochs):
      # Forward Propagation
      y_pred = self.forward_prop(X)

      # Compute Loss (optional, for monitoring)
      loss = self.compute_loss(y, y_pred)

      # Backpropagation
      dW1, db1, dW2, db2 = self.backpropagation(X, y, y_pred)

      # Gradient Descent (update weights and biases)
      self.W1 -= self.lr * dW1
      self.b1 -= self.lr * db1
      self.W2 -= self.lr * dW2
      self.b2 -= self.lr * db2

      if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss}")

# Applying ANN

In [7]:
size = X_train_prepared.shape[1]

# Target log-transformation for stability
y_train_scaled = np.log1p(y_train)

# Slightly increased learning rate and hidden layer size for better capacity
model = ANN(input_size = size, hidden_layer_size= 10, output_size= 1, lr= 0.001)

# Increased epochs to give the model more time to converge
model.training(X_train_prepared, y_train_scaled, epochs = 3000)

# Transform predictions back to original scale
y_train_pred = np.expm1(model.forward_prop(X_train_prepared))
y_test_pred = np.expm1(model.forward_prop(X_test_prepared))


print("Training complete. You can now re-run the evaluation cell to see the improved metrics.")

Epoch 0, Loss: 32.16236236675985
Epoch 100, Loss: 27.972123628234783
Epoch 200, Loss: 24.532126058253436
Epoch 300, Loss: 21.68743096081317
Epoch 400, Loss: 19.287170571674796
Epoch 500, Loss: 17.162968065927753
Epoch 600, Loss: 15.14143798057023
Epoch 700, Loss: 13.191744882001297
Epoch 800, Loss: 11.579779536735998
Epoch 900, Loss: 10.501740884371147
Epoch 1000, Loss: 9.795137787682519
Epoch 1100, Loss: 9.265169040219048
Epoch 1200, Loss: 8.828507573837985
Epoch 1300, Loss: 8.45511073407786
Epoch 1400, Loss: 8.135768720843833
Epoch 1500, Loss: 7.881413515117857
Epoch 1600, Loss: 7.69118249969121
Epoch 1700, Loss: 7.549174075203619
Epoch 1800, Loss: 7.443825319042291
Epoch 1900, Loss: 7.3677390422415225
Epoch 2000, Loss: 7.314689000193508
Epoch 2100, Loss: 7.27897912564261
Epoch 2200, Loss: 7.254802066385431
Epoch 2300, Loss: 7.238408357415034
Epoch 2400, Loss: 7.227154386780426
Epoch 2500, Loss: 7.2190000166265955
Epoch 2600, Loss: 7.21266528719968
Epoch 2700, Loss: 7.207733919699693

# Results

In [8]:
from sklearn.metrics import mean_squared_error
import numpy as np # Added numpy import

def smape(y_true, y_pred):
  num = np.abs(y_true - y_pred)
  den = (np.abs(y_true) + np.abs(y_pred)) / 2

  return np.mean(num/den) * 100

In [9]:
# Calculate RMSE (Root Mean Squared Error)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

# Calculate sMAPE
train_smape_val = smape(y_train.values.reshape(-1, 1), y_train_pred)
test_smape_val = smape(y_test.values.reshape(-1, 1), y_test_pred)

print(f"Train RMSE: {train_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Train sMAPE: {train_smape_val:.2f}%")
print(f"Test sMAPE: {test_smape_val:.2f}%")

Train RMSE: 1356494364.33
Test RMSE: 1471778302.01
Train sMAPE: 136.06%
Test sMAPE: 136.64%


#